# Course 19 lab — Bug fixes and small changes

This notebook investigates `BUG-2415` as a structured maintenance change. Run it from the `northstar-bugfix` directory. The notebook is an exploration surface; `lab.py` and the protected JSON contracts are the reproducible sources of truth.

## 1. Load the executable workflow

The reference uses only deterministic local fixtures. No credentials or network access are required.

In [ ]:
import importlib.util
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
spec = importlib.util.spec_from_file_location('course19_notebook_lab', Path('lab.py'))
course19_lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = course19_lab
spec.loader.exec_module(course19_lab)
build_candidate = course19_lab.build_candidate
build_report = course19_lab.build_report
run_triage = course19_lab.run_triage
assess_candidate = course19_lab.assess_candidate
from bugfix_app.risk_engine import classify_risk_buggy, classify_risk_fixed
from bugfix_app.workflow import select_workflow_mode


## 2. Reproduce before repairing

A green candidate is not evidence that the baseline contained the reported defect. Observe both revisions at the exact boundary.

In [ ]:
[(score, classify_risk_buggy(score), classify_risk_fixed(score)) for score in (79, 80, 81)]

## 3. Let consequence and uncertainty select the workflow

Compare a documentation typo, the known boundary defect, an authorization change, and an AI citation regression. Diff size is not an input.

In [ ]:
run_triage()

## 4. Inspect the clean handoff

The candidate proposes source files, purposes, requirement meaning, and evidence references. The trusted harness separately loads the oracle and work unit, creates real Git commits, derives the diff, executes both revisions, and resolves authenticated evidence.

In [ ]:
clean = build_candidate()
clean_decision, clean_context = assess_candidate(clean)
clean_decision, clean_context.actual_diff_entries

## 5. Try to make green easier

The same agent changes the acceptance contract so score 80 may remain MEDIUM. The independent oracle comparison must catch the content change.

In [ ]:
assess_candidate(build_candidate(['weaken_oracle']))[0]

## 6. Expand scope inside a plausible story

A decision-service edit may sound helpful, but it is outside the exact write set. A separate work unit is required.

In [ ]:
assess_candidate(build_candidate(['scope_expansion']))[0]

## 7. Test the production path

A corrected classifier cannot compensate for an upstream normalizer that changes 80 to 79. This mutation demonstrates root-cause mismatch.

In [ ]:
decision, context = assess_candidate(build_candidate(['root_cause_mismatch']))
decision, context.unit_cause_evidence.status, context.production_cause_evidence.status

## 8. Replay an old green receipt

Evidence is useful only for the subjects it evaluated. Rebinding an older candidate receipt to this change must fail.

In [ ]:
assess_candidate(build_candidate(['replay_old_evidence']))[0]

## 9. Run the complete evidence package

The course-validation result and candidate-specific assessment are separate. The suite should report `PASS` with 12/12 conformance, 30/30 evaluation, and 10/10 experiments. Only the exact reference candidate should report `READY_FOR_INTEGRATION_REVIEW`; neither result authorizes merge or deployment.

In [ ]:
report = build_report()
{
    'course_validation': report['course_validation'],
    'candidate_assessment': report['candidate_assessment'],
    'conformance': (report['conformance']['passed'], report['conformance']['total']),
    'evaluation': (report['evaluation']['passed'], report['evaluation']['total']),
    'experiments': (report['experiments']['passed'], report['experiments']['total']),
    'comparison': report['comparison'],
}

## 10. Your turn

Complete `workshop/starter/bugfix-package.json`. Then add a new mutation for a renamed protected file. Explain which trusted component derives the diff, who may revise the oracle, and which condition would escalate this change to `GOVERNED`.